# Analisis Spasial Autokorelasi (Global Moran's I)
Skrip ini dirancang secara khusus untuk memproses data titik panas (*hotspot*) dalam format *shapefile*, rentang tahun 2010 hingga 2020. Berdasarkan nilai `Gi_Bin`, data spasial ini dikelompokkan ke dalam 3 (tiga) kategori: **Hotspot** (Gi_Bin > 0), **Tidak Signifikan** (Gi_Bin = 0), dan **Coldspot** (Gi_Bin < 0).

### Alur Kerja Geokomputasi:
1. **Iterasi Temporal:** Skrip akan melakukan iterasi pada *shapefile* secara otomatis dari tahun 2010 hingga 2020.
2. **Seleksi Atribut:** Memisahkan fitur spasial menjadi tiga klasifikasi menggunakan `SelectLayerByAttribute`.
3. **Evaluasi Spasial:** Menjalankan modul *Spatial Autocorrelation* dari ArcPy. 
4. **Konversi PDF Laporan:** Skrip menangkap *path* direktori tersembunyi (*scratch folder*) berisi file HTML yang dikeluarkan oleh ArcGIS, kemudian mengonversinya menjadi PDF.
5. **Ekspor Tabular Otomatis:** Menggabungkan Moran's Index, Z-Score, dan P-Value seluruh observasi ke dalam format *spreadsheet* Excel yang sudah diberikan sentuhan desain akademik yang terstruktur.

---
### ⚠️ Persiapan Lingkungan Sistem (Environment)
Karena Anda akan melakukan konversi HTML-ke-PDF secara langsung di dalam Jupyter Notebook, dua dependensi ini wajib disiapkan:
1. **Library Python:** Instal `pdfkit`. (Buka *Python Command Prompt* milik ArcGIS Pro, ketik `pip install pdfkit` dan `pip install openpyxl`).
2. **Modul Engine:** Pasang perangkat lunak [*wkhtmltopdf*](https://wkhtmltopdf.org/downloads.html) di komputer Anda. Setelah terinstal, pastikan lokasi `wkhtmltopdf.exe` diubah pada *cell* kode di bawah ini.

In [15]:
%pip install pdfkit openpyxl --user

Note: you may need to restart the kernel to use updated packages.


In [16]:
import arcpy
import os
import re
import shutil
import urllib.parse
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment

In [17]:
# --------------------------------------------------------
# 1. PENGATURAN LINGKUNGAN KERJA & PDF ENGINE
# --------------------------------------------------------
arcpy.env.overwriteOutput = True

base_dir = r"D:\Skripsi\03_Data_Hasil\Titik Panas (Hotspot)"
input_folder = os.path.join(base_dir, "Hotspot Clean Jambi")
output_folder = os.path.join(base_dir, "Morans Index")

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

try:
    import pdfkit
    path_wkhtmltopdf = r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe"
    if os.path.exists(path_wkhtmltopdf):
        config = pdfkit.configuration(wkhtmltopdf=path_wkhtmltopdf)
        pdf_engine_ready = True
    else:
        pdf_engine_ready = False
except ImportError:
    pdf_engine_ready = False

# Fungsi internal untuk ekstraksi metrik dari HTML
def proses_ekstraksi_html(result_obj):
    arcpy_html_path = None
    for i in range(result_obj.outputCount):
        out_val = str(result_obj.getOutput(i))
        if out_val.lower().endswith('.html'):
            arcpy_html_path = out_val
            break
            
    if not arcpy_html_path:
        msg = result_obj.getMessages(0)
        html_match = re.search(r'([A-Za-z]:[\\/][^\n*?"<>|]+\.html)', msg)
        if html_match:
            arcpy_html_path = html_match.group(1).strip()
            
    if arcpy_html_path:
        if arcpy_html_path.startswith("file:///"):
            arcpy_html_path = arcpy_html_path.replace("file:///", "")
        arcpy_html_path = urllib.parse.unquote(arcpy_html_path)
        arcpy_html_path = os.path.normpath(arcpy_html_path)

    moran_idx, z_scr, p_val = None, None, None

    if arcpy_html_path and os.path.exists(arcpy_html_path):
        with open(arcpy_html_path, 'r', encoding='utf-8', errors='ignore') as f:
            html_content = f.read()
        
        clean_text = re.sub(r'<[^>]+>', ' ', html_content)
        
        def ambil_angka(patterns, text):
            for pattern in patterns:
                match = re.search(pattern, text, re.IGNORECASE)
                if match:
                    return float(match.group(1).replace(',', '.'))
            return None
        
        moran_idx = ambil_angka([r"Moran's Index[\s:]*([-\d.,]+)", r"Indeks Moran[\s:]*([-\d.,]+)"], clean_text)
        z_scr = ambil_angka([r"z-score[\s:]*([-\d.,]+)", r"Skor-z[\s:]*([-\d.,]+)"], clean_text)
        p_val = ambil_angka([r"p-value[\s:]*([-\d.,]+)", r"Nilai-p[\s:]*([-\d.,]+)"], clean_text)
        
    return arcpy_html_path, moran_idx, z_scr, p_val

In [18]:
# --------------------------------------------------------
# 2. GEOKOMPUTASI AUTOKORELASI SPASIAL (2010-2020)
# --------------------------------------------------------
years = range(2010, 2021)
categories = {
    "Hotspot": "Gi_Bin > 0",
    "Tidak_Signifikan": "Gi_Bin = 0",
    "Coldspot": "Gi_Bin < 0"
}

summary_data_tahunan = []
summary_data_kategori = []

print("=== MEMULAI KOMPUTASI AUTOKORELASI SPASIAL ===")

for year in years:
    shp_name = f"Hotspot_{year}_03_Gi.shp"
    shp_path = os.path.join(input_folder, shp_name)
    
    if not arcpy.Exists(shp_path):
        continue
        
    print(f"\n[ Tahun {year} ]")
    
    # ----------------------------------------------------------------
    # FASE A: EVALUASI KESELURUHAN (TANPA REKLASIFIKASI)
    # ----------------------------------------------------------------
    print(f"   -> Evaluasi Keseluruhan (Tahunan)")
    count_total = int(arcpy.management.GetCount(shp_path)[0])
    
    if count_total < 30:
        print(f"      [!] Observasi terbatas (n = {count_total}). Dilewati.")
    else:
        try:
            result_total = arcpy.stats.SpatialAutocorrelation(
                Input_Feature_Class=shp_path,
                Input_Field="Gi_Bin",
                Generate_Report="GENERATE_REPORT",
                Conceptualization_of_Spatial_Relationships="INVERSE_DISTANCE",
                Distance_Method="EUCLIDEAN_DISTANCE",
                Standardization="ROW"
            )
            
            html_path_tot, moran_tot, z_tot, p_tot = proses_ekstraksi_html(result_total)
            
            if moran_tot is not None and z_tot is not None and p_tot is not None:
                summary_data_tahunan.append({
                    "Tahun": year,
                    "Populasi Titik (n)": count_total,
                    "Moran's Index": round(moran_tot, 6),
                    "Z-Score": round(z_tot, 6),
                    "P-Value": round(p_tot, 6)
                })
                
                if html_path_tot:
                    if pdf_engine_ready:
                        pdf_name_tot = f"Moran's - {year} - Keseluruhan.pdf"
                        pdf_kit_path = os.path.join(output_folder, pdf_name_tot)
                        pdfkit.from_file(html_path_tot, pdf_kit_path, configuration=config)
                        print(f"      [v] PDF Tersimpan: {pdf_name_tot}")
                    else:
                        html_name_tot = f"Moran's - {year} - Keseluruhan.html"
                        html_dest_tot = os.path.join(output_folder, html_name_tot)
                        shutil.copy2(html_path_tot, html_dest_tot)
                        print(f"      [v] HTML Tersimpan: {html_name_tot}")
            else:
                print("      [X] Ekstraksi dari dokumen HTML gagal.")
                
        except Exception as e:
            err_msg = str(e)
            if "000906" in err_msg or "000871" in err_msg:
                print("      [X] Varians Nol: Nilai seluruh titik persis sama.")
            else:
                print(f"      [X] Terjadi anomali: {err_msg}")

    # ----------------------------------------------------------------
    # FASE B: EVALUASI BERDASARKAN KATEGORI (SPLIT VIRTUAL)
    # ----------------------------------------------------------------
    for cat_name, where_clause in categories.items():
        print(f"   -> Evaluasi Kategori: {cat_name.replace('_', ' ')}")
        
        temp_fc = rf"memory\{cat_name}_{year}"
        
        try:
            arcpy.analysis.Select(shp_path, temp_fc, where_clause)
            count_cat = int(arcpy.management.GetCount(temp_fc)[0])
            
            if count_cat < 30:
                print(f"      [!] Observasi terbatas (n = {count_cat}). Dilewati.")
                arcpy.management.Delete(temp_fc)
                continue
                
            if cat_name == "Tidak_Signifikan":
                print(f"      [!] Kategori homogen (seluruh nilai = 0). Dilewati.")
                arcpy.management.Delete(temp_fc)
                continue
                
            result_cat = arcpy.stats.SpatialAutocorrelation(
                Input_Feature_Class=temp_fc,
                Input_Field="Gi_Bin",
                Generate_Report="GENERATE_REPORT",
                Conceptualization_of_Spatial_Relationships="INVERSE_DISTANCE",
                Distance_Method="EUCLIDEAN_DISTANCE",
                Standardization="ROW"
            )
            
            html_path_cat, moran_cat, z_cat, p_cat = proses_ekstraksi_html(result_cat)
            
            if moran_cat is not None and z_cat is not None and p_cat is not None:
                summary_data_kategori.append({
                    "Tahun": year,
                    "Kategori": cat_name.replace("_", " "),
                    "Populasi Titik (n)": count_cat,
                    "Moran's Index": round(moran_cat, 6),
                    "Z-Score": round(z_cat, 6),
                    "P-Value": round(p_cat, 6)
                })
                
                if html_path_cat:
                    if pdf_engine_ready:
                        pdf_name_cat = f"Moran's - {year} - {cat_name}.pdf"
                        pdf_kit_path_cat = os.path.join(output_folder, pdf_name_cat)
                        pdfkit.from_file(html_path_cat, pdf_kit_path_cat, configuration=config)
                        print(f"      [v] PDF Tersimpan: {pdf_name_cat}")
                    else:
                        html_name_cat = f"Moran's - {year} - {cat_name}.html"
                        html_dest_cat = os.path.join(output_folder, html_name_cat)
                        shutil.copy2(html_path_cat, html_dest_cat)
                        print(f"      [v] HTML Tersimpan: {html_name_cat}")
            else:
                print("      [X] Ekstraksi dari dokumen HTML gagal.")

        except Exception as e:
            err_msg = str(e)
            if "000906" in err_msg or "000871" in err_msg:
                print("      [X] Varians Nol: Algoritma membatalkan operasi.")
            else:
                print(f"      [X] Terjadi anomali teknis: {err_msg}")
            
        finally:
            if arcpy.Exists(temp_fc):
                arcpy.management.Delete(temp_fc)

=== MEMULAI KOMPUTASI AUTOKORELASI SPASIAL ===

[ Tahun 2010 ]
   -> Evaluasi Keseluruhan (Tahunan)
      [v] HTML Tersimpan: Moran's - 2010 - Keseluruhan.html
   -> Evaluasi Kategori: Hotspot
      [!] Observasi terbatas (n = 16). Dilewati.
   -> Evaluasi Kategori: Tidak Signifikan
      [!] Kategori homogen (seluruh nilai = 0). Dilewati.
   -> Evaluasi Kategori: Coldspot
      [!] Observasi terbatas (n = 1). Dilewati.

[ Tahun 2011 ]
   -> Evaluasi Keseluruhan (Tahunan)
      [v] HTML Tersimpan: Moran's - 2011 - Keseluruhan.html
   -> Evaluasi Kategori: Hotspot
      [v] HTML Tersimpan: Moran's - 2011 - Hotspot.html
   -> Evaluasi Kategori: Tidak Signifikan
      [!] Kategori homogen (seluruh nilai = 0). Dilewati.
   -> Evaluasi Kategori: Coldspot
      [v] HTML Tersimpan: Moran's - 2011 - Coldspot.html

[ Tahun 2012 ]
   -> Evaluasi Keseluruhan (Tahunan)
      [v] HTML Tersimpan: Moran's - 2012 - Keseluruhan.html
   -> Evaluasi Kategori: Hotspot
      [v] HTML Tersimpan: Moran's - 2

In [20]:
# --------------------------------------------------------
# 3. KOMPILASI TABULAR KE SPREADSHEET (MULTI-SHEET)
# --------------------------------------------------------
excel_path = os.path.join(output_folder, "Rekapitulasi_Indeks_Moran_2010_2020.xlsx")
data_to_export = {}

if summary_data_tahunan:
    data_to_export["Rekap_Tahunan"] = pd.DataFrame(summary_data_tahunan)
if summary_data_kategori:
    data_to_export["Rekap_Kategori"] = pd.DataFrame(summary_data_kategori)

if data_to_export:
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for sheet_name, df in data_to_export.items():
            df.to_excel(writer, index=False, sheet_name=sheet_name)
            worksheet = writer.sheets[sheet_name]
            
            # Formatting Akademik
            header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
            header_font = Font(color="FFFFFF", bold=True)
            align_center = Alignment(horizontal="center", vertical="center")
            
            for cell in worksheet[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = align_center
                
            # Autofit Columns
            for col in worksheet.columns:
                max_length = 0
                col_letter = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                worksheet.column_dimensions[col_letter].width = max_length + 4
                
    print(f"\n=== EKSEKUSI GEOKOMPUTASI SELESAI ===")
    print(f"File rekapitulasi (Multi-Sheet) tersimpan di: {excel_path}")
else:
    print("\n=== EKSEKUSI SELESAI ===")
    print("Tidak ada data metrik spasial yang berhasil direkam.")


=== EKSEKUSI GEOKOMPUTASI SELESAI ===
File rekapitulasi (Multi-Sheet) tersimpan di: D:\Skripsi\03_Data_Hasil\Titik Panas (Hotspot)\Morans Index\Rekapitulasi_Indeks_Moran_2010_2020.xlsx
